# 🧉 Voz argentina para Mate (Piper)

Entrena una voz **masculina argentina** para Mate, gratis, que después corre **offline y al instante** en cualquier compu.

- Datos: *Crowdsourced high-quality Argentinian Spanish speech data set* de Google (OpenSLR 61, licencia **CC BY-SA 4.0**), hablante 05223.
- Base: la voz masculina **Aldo** de Piper (es_MX, calidad *medium*): se arranca con sus pesos y se la reentrena con la voz argentina.

**Cómo usarlo**
1. Arriba: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*.
2. En tu Google Drive creá la carpeta `mate_voz` y subí ahí `dataset_mate.zip` (Mate te lo dejó en el Escritorio).
3. Corré las celdas en orden (▶). El entrenamiento guarda en Drive: si Colab se corta, volvé a correr todo y **sigue desde donde quedó**.
4. Cuando quieras escuchar cómo va: corré las celdas 6 y 7 (no hace falta esperar al final).
5. Al final bajá de `mate_voz` los archivos `es_AR-mate-medium.onnx` y `es_AR-mate-medium.onnx.json` y ponelos en `C:\Users\<vos>\.mate\voces`. En Mate → Ajustes → Voz aparece la nueva.

In [ ]:
# 1. Drive y carpeta de trabajo
from google.colab import drive
drive.mount('/content/drive')
import os, glob
W = '/content/drive/MyDrive/mate_voz'
os.makedirs(W, exist_ok=True)
assert os.path.exists(f'{W}/dataset_mate.zip'), 'Falta subir dataset_mate.zip a la carpeta mate_voz de tu Drive'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Instalar Piper con entrenamiento (docs/TRAINING.md de OHF-Voice/piper1-gpl) — tarda unos minutos
!apt-get -qq install -y build-essential cmake ninja-build > /dev/null
!rm -rf /content/piper1-gpl && git clone -q https://github.com/OHF-Voice/piper1-gpl.git /content/piper1-gpl
%cd /content/piper1-gpl
!python3 -m pip -q install -e '.[train]'
!bash ./build_monotonic_align.sh
!python3 setup.py build_ext --inplace > /dev/null
!python3 -c "import piper.train; print('piper train OK')"

In [ ]:
# 3. Datos: 143 frases de la voz elegida, ya cortadas, a 22050 Hz y emparejadas con su texto (formato archivo.wav|texto)
!rm -rf /content/data && mkdir -p /content/data
!unzip -q -o {W}/dataset_mate.zip -d /content/data
!echo "Frases: $(wc -l < /content/data/metadata.csv)" && head -3 /content/data/metadata.csv

In [ ]:
# 4. Voz base (Aldo, es_MX medium) para no empezar de cero
BASE = f'{W}/base_aldo.ckpt'
if not os.path.exists(BASE):
    !wget -q -O {BASE} 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es/es_MX/ald/medium/epoch%3D3799-step%3D114000.ckpt'
print(round(os.path.getsize(BASE) / 1e6), 'MB')

In [ ]:
# 5. Entrenar. Guarda en Drive; si Colab se corta, volvé a correr esta celda y retoma solo.
#    1ª vez: arranca con los pesos de Aldo (--model.warmstart_ckpt). Después: sigue del último guardado (--ckpt_path).
#    2000 épocas es lo que recomienda Piper para afinar con menos de 5 h de audio (acá es poco audio: va rápido).
prev = sorted(glob.glob(f'{W}/checkpoints/**/*.ckpt', recursive=True), key=os.path.getmtime)
desde = f'--ckpt_path "{prev[-1]}"' if prev else f'--model.warmstart_ckpt "{BASE}"'
print('Arranco:', desde)
!python3 -m piper.train fit \
  --data.voice_name es_AR-mate \
  --data.csv_path /content/data/metadata.csv \
  --data.audio_dir /content/data/wav/ \
  --model.sample_rate 22050 \
  --data.espeak_voice es-419 \
  --data.cache_dir /content/cache/ \
  --data.config_path {W}/es_AR-mate-medium.onnx.json \
  --data.batch_size 16 \
  --trainer.max_epochs 2000 \
  --trainer.default_root_dir {W}/checkpoints \
  {desde}

In [ ]:
# 6. Exportar la voz para Mate (se puede correr en cualquier momento para escuchar cómo va)
last = sorted(glob.glob(f'{W}/checkpoints/**/*.ckpt', recursive=True), key=os.path.getmtime)[-1]
print('Exporto:', last)
!python3 -m piper.train.export_onnx --checkpoint "{last}" --output-file {W}/es_AR-mate-medium.onnx
!ls -la {W}/es_AR-mate-medium.onnx*

In [ ]:
# 7. Escuchar una prueba
!python3 -m pip -q install piper-tts
import wave
from piper import PiperVoice
v = PiperVoice.load(f'{W}/es_AR-mate-medium.onnx')
with wave.open('/content/prueba.wav', 'wb') as w:
    v.synthesize_wav('¡Buenas! Soy Mate. Ya te anoté llamar al contador para el jueves. ¿Querés que te avise un rato antes?', w)
from IPython.display import Audio
Audio('/content/prueba.wav')

---
Atribución de los datos: *Guevara-Rukoz et al., Crowdsourcing Latin American Spanish for Low-Resource Text-to-Speech, LREC 2020* — Google, OpenSLR 61, CC BY-SA 4.0. La voz resultante se comparte con la misma licencia.